# Supervised Classification Modeling & Hyperparameter Tuning

**Project:** Compare four classification architectures, tune them with stratified cross-validation, evaluate a held-out test set, and save the best model.

## Dataset
Adult Census Income from OpenML (mixed numerical and categorical columns). The target is whether income exceeds `$50K`.

## Models compared
1. Logistic Regression
2. Random Forest
3. Extra Trees
4. HistGradientBoosting

All four models are from Scikit-Learn, so the notebook can run without optional XGBoost/LightGBM installations. XGBoost or LightGBM can be added later if installed.

## Leakage-safe experiment design
- Split raw data into train and test sets before fitting preprocessing.
- Keep imputation, scaling, and encoding inside a `Pipeline`.
- Use `StratifiedKFold` inside `GridSearchCV` on training data only.
- Choose the champion using mean cross-validation ROC-AUC.
- Evaluate the champion once on the held-out test set.
- Save the complete fitted pipeline with `joblib`.

**First run requires internet access** to download Adult from OpenML.

## 1. Install/import libraries

In [ ]:
# If needed in Google Colab, uncomment:
# !pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)

RANDOM_STATE = 42
POSITIVE_LABEL = ">50K"
sns.set_theme(style="whitegrid")

## 2. Load the dataset

In [ ]:
adult = fetch_openml(name="adult", version=2, as_frame=True)
df = adult.frame.copy()
target_column = "class"

# OpenML may represent missing values as '?'.
df = df.replace("?", np.nan)

X = df.drop(columns=[target_column])
y = df[target_column].astype(str).str.strip()

print("Dataset shape:", df.shape)
print("\\nTarget counts:")
display(y.value_counts().to_frame("count"))
display(X.head())

## 3. Train/test split before preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

## 4. Build preprocessing and model pipelines

The one-hot encoder returns dense output here so all four selected estimators can consume the same transformed matrix. For very high-cardinality datasets, use sparse-compatible estimators or a different encoding strategy to reduce memory use.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

def make_pipeline(estimator):
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", estimator)
    ])

## 5. Define four model architectures and search spaces

`GridSearchCV` optimizes ROC-AUC using stratified 3-fold cross-validation. The grids are intentionally compact to keep this educational experiment manageable. Increase them for a more exhaustive search.

In [ ]:
model_grids = {
    "Logistic Regression": (
        LogisticRegression(max_iter=1500, class_weight="balanced", random_state=RANDOM_STATE),
        {
            "classifier__C": [0.1, 1.0, 5.0],
            "classifier__solver": ["lbfgs"]
        }
    ),
    "Random Forest": (
        RandomForestClassifier(
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=1
        ),
        {
            "classifier__n_estimators": [150, 250],
            "classifier__max_depth": [None, 18],
            "classifier__min_samples_leaf": [1, 2]
        }
    ),
    "Extra Trees": (
        ExtraTreesClassifier(
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=1
        ),
        {
            "classifier__n_estimators": [150, 250],
            "classifier__max_depth": [None, 18],
            "classifier__min_samples_leaf": [1, 2]
        }
    ),
    "HistGradientBoosting": (
        HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            "classifier__learning_rate": [0.05, 0.1],
            "classifier__max_iter": [100, 160],
            "classifier__max_leaf_nodes": [15, 31]
        }
    )
}

cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

## 6. Tune each model with GridSearchCV

Each search sees only `X_train` and `y_train`. Since preprocessing is inside the pipeline, imputers, scalers, and encoders are refitted separately inside each CV training fold.

In [ ]:
searches = {}
search_rows = []

for model_name, (estimator, param_grid) in model_grids.items():
    print(f"\nTuning: {model_name}")

    search = GridSearchCV(
        estimator=make_pipeline(estimator),
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv_strategy,
        n_jobs=-1,
        refit=True,
        verbose=1,
        return_train_score=False
    )
    search.fit(X_train, y_train)

    searches[model_name] = search
    search_rows.append({
        "model": model_name,
        "best_cv_roc_auc": search.best_score_,
        "best_parameters": search.best_params_
    })

cv_results = pd.DataFrame(search_rows).sort_values(
    "best_cv_roc_auc", ascending=False
).reset_index(drop=True)

display(cv_results)

## 7. Select the champion using validation performance

The champion is the model with the highest mean cross-validation ROC-AUC. The held-out test set has not been used to select or tune the model.

In [ ]:
champion_name = cv_results.loc[0, "model"]
champion_search = searches[champion_name]
champion_model = champion_search.best_estimator_

print("Champion model:", champion_name)
print("Best mean CV ROC-AUC:", round(champion_search.best_score_, 4))
print("Best parameters:")
print(champion_search.best_params_)

## 8. Evaluate every tuned model on the held-out test set

Report accuracy, precision, recall, F1-score, and ROC-AUC. The positive class is income `>50K`.

In [ ]:
test_rows = []
test_probabilities = {}

for model_name, search in searches.items():
    fitted_model = search.best_estimator_
    predictions = fitted_model.predict(X_test)
    classes = list(fitted_model.named_steps["classifier"].classes_)
    positive_index = classes.index(POSITIVE_LABEL)
    probabilities = fitted_model.predict_proba(X_test)[:, positive_index]
    test_probabilities[model_name] = probabilities

    test_rows.append({
        "model": model_name,
        "cv_roc_auc": search.best_score_,
        "test_accuracy": accuracy_score(y_test, predictions),
        "test_precision": precision_score(y_test, predictions, pos_label=POSITIVE_LABEL, zero_division=0),
        "test_recall": recall_score(y_test, predictions, pos_label=POSITIVE_LABEL, zero_division=0),
        "test_f1": f1_score(y_test, predictions, pos_label=POSITIVE_LABEL, zero_division=0),
        "test_roc_auc": roc_auc_score((y_test == POSITIVE_LABEL).astype(int), probabilities)
    })

comparison = pd.DataFrame(test_rows).sort_values(
    "cv_roc_auc", ascending=False
).reset_index(drop=True)

display(comparison.round(4))

## 9. Compare ROC-AUC curves

In [ ]:
y_test_binary = (y_test == POSITIVE_LABEL).astype(int)

plt.figure(figsize=(9, 7))
for model_name, probabilities in test_probabilities.items():
    fpr, tpr, _ = roc_curve(y_test_binary, probabilities)
    auc_value = roc_auc_score(y_test_binary, probabilities)
    plt.plot(fpr, tpr, label=f"{model_name} (AUC = {auc_value:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Held-out Test Set")
plt.legend()
plt.tight_layout()
plt.savefig("roc_auc_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Champion confusion matrix and classification report

In [ ]:
champion_predictions = champion_model.predict(X_test)

print(classification_report(
    y_test,
    champion_predictions,
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    champion_predictions,
    labels=[">50K", "<=50K"],
    xticks_rotation=30,
    cmap="Blues"
)
plt.title(f"Champion Confusion Matrix — {champion_name}")
plt.tight_layout()
plt.savefig("champion_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Save the champion model and result files

The serialized file contains the **complete fitted pipeline**, including preprocessing and classifier. Use the same raw input columns when loading it for future predictions.

In [ ]:
MODEL_PATH = "champion_model.joblib"
joblib.dump(champion_model, MODEL_PATH)

comparison.to_csv("model_comparison.csv", index=False)

print("Saved model:", MODEL_PATH)
print("Saved comparison table: model_comparison.csv")
print("Saved ROC plot: roc_auc_comparison.png")
print("Saved confusion matrix: champion_confusion_matrix.png")

## 12. Load the saved model and make predictions

This verifies that the saved artifact can be loaded and used without manually preprocessing the input.

In [ ]:
loaded_model = joblib.load("champion_model.joblib")

sample_rows = X_test.head(5)
sample_predictions = loaded_model.predict(sample_rows)
sample_probabilities = loaded_model.predict_proba(sample_rows)

display(pd.DataFrame({
    "actual": y_test.head(5).values,
    "predicted": sample_predictions
}, index=sample_rows.index))

## 13. Submission checklist

- [x] Four distinct model architectures
- [x] Hyperparameter optimization with `GridSearchCV`
- [x] Stratified 3-fold cross-validation
- [x] Preprocessing fitted only inside training folds
- [x] Precision, Recall, F1-score, and ROC-AUC
- [x] Model comparison table
- [x] ROC-AUC curves
- [x] Champion confusion matrix
- [x] Champion selected using cross-validation ROC-AUC
- [x] Code to serialize the full champion pipeline as `champion_model.joblib`

### Important
Run the notebook from top to bottom before submission. The `.joblib`, `.csv`, and `.png` output files are created by the notebook during execution; upload those generated files to GitHub along with this notebook.